# Week 6 Session 1: SQL fundamentals at Sunrise Stores
**Forge Academy | Data Science Foundations**

*Covers Week 6 Session 1: SQL fundamentals. We use SQLite inside this notebook, so you do not need to install a database server.*

## Mission

Sunrise Stores is a small provisions shop. The owner, Ngozi, has been keeping five tables: customers, products, orders, order items and payments. She can open the files, but she cannot yet ask them a precise question, such as "how many of last season's orders were cancelled?" or "which product category has the highest average price?"

Today we learn enough SQL to ask those questions of one table at a time. Next session we will join the tables and start the SQL Analytics Challenge.

Keep `sunrise_stores.db` in the same folder as this notebook, open the folder in VS Code, and run the cells from top to bottom.


## 0. What SQL is, in plain words

- A **database** is a collection of tables. Each table has a name, columns, and rows, just like a CSV, but we talk to it with a language called **SQL** (Structured Query Language).
- A **query** is a question written in SQL. The most common pattern is: take rows `FROM` a table, keep only some of them with `WHERE`, and `SELECT` the columns (or the totals) you want to see, then `ORDER BY` the result.
- We will run SQL from this notebook with `sqlite3` (part of Python) and `pandas` (to display the result as a table). The ideas you learn will still hold if you later use another database tool.

The shop's five tables, and the grain of each one (what one row means):

| Table | One row is | Useful columns |
|---|---|---|
| `customers` | one person who has signed up | `customer_id`, `first_name`, `last_name`, `city`, `signup_date`, `segment` |
| `products` | one item on the shelf | `product_id`, `product_name`, `category`, `unit_price`, `unit_cost` |
| `orders` | one basket / checkout | `order_id`, `customer_id`, `order_date`, `status`, `channel` |
| `order_items` | one line on an order (a product and a quantity) | `item_id`, `order_id`, `product_id`, `quantity`, `unit_price` |
| `payments` | one payment against an order | `payment_id`, `order_id`, `payment_date`, `method`, `amount` |

`segment` is `New`, `Regular` or `Wholesale`. `channel` is `Store`, `WhatsApp` or `Website`. `status` is `Completed`, `Cancelled` or `Refunded`. We will treat **Completed** as a real sale when we talk about money.

The five tables are also saved as CSVs next to the database, if you ever want to peek with `pd.read_csv`. Today we query the database.


## 1. Connect to the database and look around

We import `sqlite3` and `pandas`, then open a connection to the file.


In [ ]:
import sqlite3


In [ ]:
import pandas as pd


In [ ]:
conn = sqlite3.connect("sunrise_stores.db")


If you get an error that the file could not be opened, the notebook is not running in the same folder as `sunrise_stores.db`. Move the file there, or in VS Code use **File, Open Folder** on the Week 6 folder, and run the cell again.

`sqlite_master` is a built-in list of everything in the database. This query asks for the table names.


In [ ]:
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name;", conn)


In [ ]:
tables


A first peek at each table. `LIMIT 5` means 'only the first 5 rows', which is safer than asking for everything.


In [ ]:
pd.read_sql("SELECT * FROM customers LIMIT 5;", conn)


In [ ]:
pd.read_sql("SELECT * FROM products LIMIT 5;", conn)


In [ ]:
pd.read_sql("SELECT * FROM orders LIMIT 5;", conn)


In [ ]:
pd.read_sql("SELECT * FROM order_items LIMIT 5;", conn)


In [ ]:
pd.read_sql("SELECT * FROM payments LIMIT 5;", conn)


How big is each table? `COUNT(*)` counts rows.


In [ ]:
query = """
SELECT 'customers' AS table_name, COUNT(*) AS n_rows FROM customers
UNION ALL
SELECT 'products', COUNT(*) FROM products
UNION ALL
SELECT 'orders', COUNT(*) FROM orders
UNION ALL
SELECT 'order_items', COUNT(*) FROM order_items
UNION ALL
SELECT 'payments', COUNT(*) FROM payments;
"""


In [ ]:
pd.read_sql(query, conn)


You should see 80 customers, 31 products, 420 orders, 1,071 order items and 379 payments. We will keep coming back to those counts. If a later query quietly changes the number of rows, something about the question (or a join, next session) needs a second look.


## 2. SELECT, ORDER BY and a first filter

You rarely need every column. Name the ones you want. `ORDER BY` sorts the result; add `DESC` for highest first.


In [ ]:
query = """
SELECT product_name, category, unit_price
FROM products
ORDER BY unit_price DESC;
"""


In [ ]:
pd.read_sql(query, conn)


`WHERE` keeps only the rows that match a condition. Text values go in single quotes. This lists the beverages, expensive first.


In [ ]:
query = """
SELECT product_name, unit_price
FROM products
WHERE category = 'Beverages'
ORDER BY unit_price DESC;
"""


In [ ]:
result = pd.read_sql(query, conn)


In [ ]:
result


You can combine conditions with `AND` and `OR`, and compare numbers with `>`, `<`, `>=`, `<=`, `=`. `BETWEEN` is inclusive on both ends. `IN` checks a list of values.


In [ ]:
query = """
SELECT product_name, category, unit_price
FROM products
WHERE unit_price >= 4000
ORDER BY unit_price DESC;
"""


In [ ]:
pd.read_sql(query, conn)


In [ ]:
query = """
SELECT first_name, last_name, city, segment
FROM customers
WHERE city IN ('Lagos', 'Ibadan')
  AND segment = 'Wholesale'
ORDER BY city, last_name;
"""


In [ ]:
pd.read_sql(query, conn)


Dates in this database are stored as text in `YYYY-MM-DD` form, so we can compare them as text and the order will still be chronological.


In [ ]:
query = """
SELECT first_name, last_name, city, signup_date
FROM customers
WHERE signup_date >= '2026-01-01'
ORDER BY signup_date
LIMIT 8;
"""


In [ ]:
pd.read_sql(query, conn)


### Your turn 1

1. List every product in the `Staples` category. Show `product_name` and `unit_price`, cheapest first.
2. How many customers live in Lagos? (Hint: `COUNT(*)` and a `WHERE` on `city`.)


In [ ]:
# Your code here


In [ ]:
# Your code here


### Your turn 2

Show `customer_id`, `first_name`, `last_name`, `signup_date` and `segment` for customers who live in Lagos and who signed up on or after 1 January 2026 and before 1 July 2026. Order the rows by `signup_date`.


In [ ]:
# Your code here


## 3. Totals: COUNT, SUM, AVG, MIN, MAX

With no `GROUP BY`, an aggregate function summarises the **whole** (filtered) table into one row.


In [ ]:
query = """
SELECT
    COUNT(*) AS n_orders,
    MIN(order_date) AS first_order,
    MAX(order_date) AS last_order
FROM orders;
"""


In [ ]:
pd.read_sql(query, conn)


How many orders are there in each status? We will group in a moment. First, filter to one status at a time so the idea of a subset is clear.


In [ ]:
query = """
SELECT COUNT(*) AS n_completed
FROM orders
WHERE status = 'Completed';
"""


In [ ]:
pd.read_sql(query, conn)


On the products table, what is the average, cheapest and most expensive unit price?


In [ ]:
query = """
SELECT
    COUNT(*) AS n_products,
    ROUND(AVG(unit_price), 1) AS avg_price,
    MIN(unit_price) AS min_price,
    MAX(unit_price) AS max_price
FROM products;
"""


In [ ]:
pd.read_sql(query, conn)


`ROUND(..., 1)` keeps one decimal place so the average is readable. `DISTINCT` counts unique values only. This is how many cities appear in the customer list, and how many different segments.


In [ ]:
query = """
SELECT
    COUNT(*) AS n_customers,
    COUNT(DISTINCT city) AS n_cities,
    COUNT(DISTINCT segment) AS n_segments
FROM customers;
"""


In [ ]:
pd.read_sql(query, conn)


## 4. GROUP BY: one total per group

`GROUP BY city` says: split the customers into a pile per city, then count each pile. Every column in the `SELECT` list must either be in the `GROUP BY` or be an aggregate (`COUNT`, `SUM`, `AVG`, ...).


In [ ]:
query = """
SELECT city, COUNT(*) AS n_customers
FROM customers
GROUP BY city
ORDER BY n_customers DESC;
"""


In [ ]:
pd.read_sql(query, conn)


The same idea, for orders by status and for products by category.


In [ ]:
query = """
SELECT status, COUNT(*) AS n_orders
FROM orders
GROUP BY status
ORDER BY n_orders DESC;
"""


In [ ]:
pd.read_sql(query, conn)


In [ ]:
query = """
SELECT
    category,
    COUNT(*) AS n_products,
    ROUND(AVG(unit_price), 1) AS avg_price,
    MIN(unit_price) AS min_price,
    MAX(unit_price) AS max_price
FROM products
GROUP BY category
ORDER BY avg_price DESC;
"""


In [ ]:
pd.read_sql(query, conn)


### Your turn 3

1. Count the customers in each `segment`. Show the segment and the count, largest group first.
2. Count the orders in each `channel`. Show the channel and the count, largest first.


In [ ]:
# Your code here


In [ ]:
# Your code here


## 5. WHERE versus HAVING

This is the idea that trips people up most often, so we will go slowly.

- `WHERE` filters **rows**, before the groups are built.
- `HAVING` filters **groups**, after `GROUP BY` has produced them.

Example: "which categories have an average unit price above 2,500?" The average does not exist until after we group, so the condition belongs in `HAVING`.


In [ ]:
query = """
SELECT
    category,
    COUNT(*) AS n_products,
    ROUND(AVG(unit_price), 1) AS avg_price
FROM products
GROUP BY category
HAVING AVG(unit_price) > 2500
ORDER BY avg_price DESC;
"""


In [ ]:
pd.read_sql(query, conn)


If you move that condition into `WHERE`, SQLite will raise an error, because `WHERE` cannot see `AVG(unit_price)` yet. That error is a useful one. The rule of thumb: if the condition is about a total or an average of a group, it is a `HAVING`. If it is about a single row (this product is a staple; this order is completed), it is a `WHERE`.

You will often use both in the same query. `WHERE` first narrows the rows, then `GROUP BY` builds the piles, then `HAVING` keeps some of the piles.


In [ ]:
query = """
SELECT city, COUNT(*) AS n_customers
FROM customers
WHERE segment = 'Regular'
GROUP BY city
HAVING COUNT(*) >= 3
ORDER BY n_customers DESC;
"""


In [ ]:
pd.read_sql(query, conn)


### Your turn 4

1. Which cities have **more than 10** customers? Show the city and the count, largest first.
2. Looking only at **Completed** orders, count the orders in each channel and keep channels that have **more than 80** completed orders. (You will need both `WHERE` and `HAVING`.)


In [ ]:
# Your code here


In [ ]:
# Your code here


## 6. Challenge (10 to 15 minutes)

Write three queries, one for each question. Use only the single tables we have already used. No joins yet.

1. List every Staples product that costs **3,000 or more**. Show `product_name` and `unit_price`, most expensive first.
2. How many orders were placed in each `status`? You already saw this pattern; write it yourself from scratch.
3. Which product categories have **more than 4** products? Show the category, the number of products, and the average `unit_price` rounded to one decimal, largest average first.


In [ ]:
# Your code here


In [ ]:
# Your code here


In [ ]:
# Your code here


## Recap

- A SQL query is a precise question: `SELECT` what you want, `FROM` a table, `WHERE` the rows you keep, `GROUP BY` when you want one total per group, `HAVING` when you filter those groups, `ORDER BY` to sort, `LIMIT` to peek.
- `COUNT`, `SUM`, `AVG`, `MIN` and `MAX` summarise. With no `GROUP BY`, they summarise the whole filtered table. With `GROUP BY`, they summarise each pile.
- `WHERE` filters rows; `HAVING` filters groups. If the condition uses an aggregate, it belongs in `HAVING`.
- `SELECT *` is a peek, not a finished answer. Name the columns you mean to show.
- We used SQLite through `sqlite3` and displayed results with `pd.read_sql`. Keep `conn` from the first cells; if something breaks, restart the kernel and Run All.

Next session we join these tables, check that we have not accidentally doubled any rows, and start the SQL Analytics Challenge. We will also look at the capstone brief, so you can start thinking about a problem and a dataset.


## Assignment (practice, before Session 2)

**Due:** before our next class. **Submit:** a short `.sql` file or a notebook, through the usual submission form.

Write numbered queries against `sunrise_stores.db`. Add a one-line comment above each query saying what it answers. Use `SELECT`, `WHERE`, `GROUP BY` and `HAVING` as needed. Do not join tables yet.

1. List every product in `Personal Care`, cheapest first.
2. Count the customers in each city, largest first.
3. How many orders were `Cancelled`?
4. Which segments have **more than 20** customers? Show the segment and the count.
5. Among products that cost **less than 2,000**, what is the average `unit_price` in each category? (Filter rows with `WHERE`, then group.)

**Rubric (20 points, 0 to 4 each)**

| Area | What earns full marks |
|---|---|
| Correctness | Queries run and the numbers are right |
| Problem solving | `WHERE` and `HAVING` are used in the right places |
| Code quality | Numbered queries, short comments, no leftover `SELECT *` |
| Communication | Each result can be read as a plain-language answer |
| Independence | You can explain every line on our call |

AI tools are allowed, but you must be able to explain every line of your work on our call.

A longer **SQL Analytics Challenge** (joins plus a write-up, 10% of the course) will be set properly in Session 2. If you want a preview, open `Week6_SQL_Analytics_Challenge.txt` in this folder, but do not worry about finishing it yet.
